# 06 - Deep Error Analysis

This notebook performs a detailed error analysis of the DenseNet-Attention model:

1. Prediction confidence distribution (correct vs. incorrect)
2. Detailed false negative and false positive analysis
3. Grad-CAM comparison between correct and misclassified samples
4. Threshold sensitivity analysis for clinical operating points

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import roc_curve

from src.data import get_dataloaders, CLASS_NAMES
from src.models import get_model
from src.evaluate import (
    evaluate_model, compute_metrics, generate_gradcam, plot_gradcam_grid,
    plot_confusion_matrix,
)
from src.utils import set_seed, get_device, load_config

plt.rcParams["figure.dpi"] = 100
sns.set_style("whitegrid")

config = load_config("../configs/default.yaml")
device = get_device()
DATA_ROOT = "../data/chest_xray"
SEED = config["reproducibility"]["seed"]

print(f"Device: {device}")

In [ ]:
set_seed(SEED)

dataloaders = get_dataloaders(
    DATA_ROOT,
    augmentation="standard",
    image_size=config["data"]["image_size"],
    batch_size=config["training"]["batch_size"],
    val_split=config["data"]["val_split"],
    num_workers=0,
    seed=SEED,
)

In [ ]:
model = get_model(
    "densenet_attention",
    pretrained=True,
    dropout=config["model"]["dropout"],
    use_attention=True,
)

checkpoint_path = "../results/checkpoints/densenet_attention_best.pt"
model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
model = model.to(device)
model.eval()

print("Model loaded from checkpoint.")

## 1. Collect all test predictions

In [ ]:
all_images, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, labels in dataloaders["test"]:
        outputs = model(images.to(device)).squeeze()
        probs = torch.sigmoid(outputs).cpu().numpy()
        all_images.append(images)
        all_labels.append(labels)
        all_probs.extend(probs.flatten())

all_images = torch.cat(all_images)
all_labels = torch.cat(all_labels).numpy()
all_probs = np.array(all_probs)
all_preds = (all_probs >= 0.5).astype(int)

print(f"Test set: {len(all_labels)} images")
print(f"  Normal:    {(all_labels == 0).sum()}")
print(f"  Pneumonia: {(all_labels == 1).sum()}")

## 2. Prediction Confidence Distribution

How confident is the model when it is right vs. when it is wrong?

In [ ]:
correct_mask = all_preds == all_labels
incorrect_mask = ~correct_mask

print(f"Correct predictions:   {correct_mask.sum()} ({correct_mask.mean()*100:.1f}%)")
print(f"Incorrect predictions: {incorrect_mask.sum()} ({incorrect_mask.mean()*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: confidence distribution split by correct/incorrect
axes[0].hist(all_probs[correct_mask], bins=50, alpha=0.6, label="Correct", color="#4C72B0", density=True)
axes[0].hist(all_probs[incorrect_mask], bins=50, alpha=0.6, label="Incorrect", color="#C44E52", density=True)
axes[0].set_xlabel("Predicted Probability (Pneumonia)")
axes[0].set_ylabel("Density")
axes[0].set_title("Confidence Distribution: Correct vs. Incorrect")
axes[0].legend()
axes[0].axvline(x=0.5, color="gray", linestyle="--", alpha=0.5)

# Right: confidence split by true class
axes[1].hist(all_probs[all_labels == 0], bins=50, alpha=0.6, label="True Normal", color="#4C72B0", density=True)
axes[1].hist(all_probs[all_labels == 1], bins=50, alpha=0.6, label="True Pneumonia", color="#DD8452", density=True)
axes[1].set_xlabel("Predicted Probability (Pneumonia)")
axes[1].set_ylabel("Density")
axes[1].set_title("Confidence Distribution by True Class")
axes[1].legend()
axes[1].axvline(x=0.5, color="gray", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("../results/confidence_distribution.png", bbox_inches="tight")
plt.show()

## 3. Detailed Error Breakdown

In [ ]:
fn_mask = (all_labels == 1) & (all_preds == 0)  # missed pneumonia
fp_mask = (all_labels == 0) & (all_preds == 1)  # false alarm
tp_mask = (all_labels == 1) & (all_preds == 1)
tn_mask = (all_labels == 0) & (all_preds == 0)

print(f"True Positives  (Pneumonia correctly detected): {tp_mask.sum()}")
print(f"True Negatives  (Normal correctly identified):  {tn_mask.sum()}")
print(f"False Negatives (Pneumonia MISSED):             {fn_mask.sum()}")
print(f"False Positives (Normal flagged as Pneumonia):  {fp_mask.sum()}")

if fn_mask.sum() > 0:
    print(f"\nFalse Negative confidence statistics:")
    fn_probs = all_probs[fn_mask]
    print(f"  Mean probability:   {fn_probs.mean():.4f}")
    print(f"  Min probability:    {fn_probs.min():.4f}")
    print(f"  Max probability:    {fn_probs.max():.4f}")
    print(f"  Borderline (0.3-0.7): {((fn_probs >= 0.3) & (fn_probs <= 0.7)).sum()}")
    print(f"  Confident wrong (<0.2): {(fn_probs < 0.2).sum()}")

if fp_mask.sum() > 0:
    print(f"\nFalse Positive confidence statistics:")
    fp_probs = all_probs[fp_mask]
    print(f"  Mean probability:   {fp_probs.mean():.4f}")
    print(f"  Min probability:    {fp_probs.min():.4f}")
    print(f"  Max probability:    {fp_probs.max():.4f}")
    print(f"  Borderline (0.3-0.7): {((fp_probs >= 0.3) & (fp_probs <= 0.7)).sum()}")
    print(f"  Confident wrong (>0.8): {(fp_probs > 0.8).sum()}")

## 4. Grad-CAM on Misclassified Samples

We compare Grad-CAM attention patterns between correctly classified and
misclassified samples to understand what the model gets wrong.

In [ ]:
import random

target_layer = model.backbone[-1]

# Select up to 4 false positives and 4 false negatives
rng = random.Random(42)

fp_indices = np.where(fp_mask)[0]
fn_indices = np.where(fn_mask)[0]

n_fp = min(4, len(fp_indices))
n_fn = min(4, len(fn_indices))

selected_fp = list(rng.sample(list(fp_indices), n_fp)) if n_fp > 0 else []
selected_fn = list(rng.sample(list(fn_indices), n_fn)) if n_fn > 0 else []
error_indices = selected_fp + selected_fn

if len(error_indices) > 0:
    error_images = all_images[error_indices]
    error_labels = all_labels[error_indices]
    error_probs = all_probs[error_indices]

    error_heatmaps = generate_gradcam(model, error_images, target_layer, device)

    fig = plot_gradcam_grid(
        error_images, error_heatmaps, error_labels, error_probs,
        n_samples=len(error_indices),
    )
    title_parts = []
    if n_fp > 0:
        title_parts.append(f"{n_fp} False Positives")
    if n_fn > 0:
        title_parts.append(f"{n_fn} False Negatives")
    plt.suptitle(f"Grad-CAM on Misclassified: {' + '.join(title_parts)}", fontsize=13, y=1.02)
    plt.savefig("../results/gradcam_errors.png", bbox_inches="tight")
    plt.show()
else:
    print("No misclassified samples at threshold 0.5.")

## 5. Grad-CAM on Correctly Classified Samples (for comparison)

We show the same number of correctly classified samples for side-by-side comparison.

In [ ]:
tp_indices = np.where(tp_mask)[0]
tn_indices = np.where(tn_mask)[0]

selected_tp = list(rng.sample(list(tp_indices), min(4, len(tp_indices))))
selected_tn = list(rng.sample(list(tn_indices), min(4, len(tn_indices))))
correct_indices = selected_tn + selected_tp

correct_images = all_images[correct_indices]
correct_labels = all_labels[correct_indices]
correct_probs = all_probs[correct_indices]

correct_heatmaps = generate_gradcam(model, correct_images, target_layer, device)

fig = plot_gradcam_grid(
    correct_images, correct_heatmaps, correct_labels, correct_probs,
    n_samples=len(correct_indices),
)
plt.suptitle("Grad-CAM on Correctly Classified (4 Normal + 4 Pneumonia)", fontsize=13, y=1.02)
plt.savefig("../results/gradcam_correct.png", bbox_inches="tight")
plt.show()

## 6. Threshold Sensitivity Analysis

How do key metrics change as we vary the decision threshold?
This is critical for choosing the right operating point in a clinical deployment.

In [ ]:
thresholds = np.arange(0.05, 0.96, 0.05)
threshold_metrics = []

for t in thresholds:
    m = compute_metrics(all_labels, all_probs, threshold=t)
    threshold_metrics.append({
        "threshold": t,
        "sensitivity": m["sensitivity"],
        "specificity": m["specificity"],
        "f1_macro": m["f1_macro"],
        "precision": m["precision"],
        "npv": m["npv"],
        "fn": m["fn"],
        "fp": m["fp"],
    })

th_df = pd.DataFrame(threshold_metrics)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(th_df["threshold"], th_df["sensitivity"], "o-", label="Sensitivity", linewidth=2)
axes[0].plot(th_df["threshold"], th_df["specificity"], "s-", label="Specificity", linewidth=2)
axes[0].plot(th_df["threshold"], th_df["f1_macro"], "^-", label="F1 (macro)", linewidth=2)
axes[0].plot(th_df["threshold"], th_df["npv"], "d-", label="NPV", linewidth=2)
axes[0].set_xlabel("Threshold")
axes[0].set_ylabel("Score")
axes[0].set_title("Metrics vs. Decision Threshold")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=0.5, color="gray", linestyle="--", alpha=0.4, label="Default (0.5)")

axes[1].plot(th_df["threshold"], th_df["fn"], "o-", label="False Negatives (missed)", color="#C44E52", linewidth=2)
axes[1].plot(th_df["threshold"], th_df["fp"], "s-", label="False Positives (false alarms)", color="#DD8452", linewidth=2)
axes[1].set_xlabel("Threshold")
axes[1].set_ylabel("Count")
axes[1].set_title("Error Counts vs. Decision Threshold")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].axvline(x=0.5, color="gray", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("../results/threshold_analysis.png", bbox_inches="tight")
plt.show()

In [ ]:
# Find key operating points
fpr, tpr, roc_thresholds = roc_curve(all_labels, all_probs)
youden_j = tpr - fpr
optimal_idx = np.argmax(youden_j)
youden_t = roc_thresholds[optimal_idx]

print("Key operating points:")
print(f"\n  Default threshold (0.5):")
m_default = compute_metrics(all_labels, all_probs, threshold=0.5)
print(f"    Sensitivity: {m_default['sensitivity']:.4f}, Specificity: {m_default['specificity']:.4f}")
print(f"    FN: {m_default['fn']}, FP: {m_default['fp']}")

print(f"\n  Youden-optimal threshold ({youden_t:.4f}):")
m_youden = compute_metrics(all_labels, all_probs, threshold=youden_t)
print(f"    Sensitivity: {m_youden['sensitivity']:.4f}, Specificity: {m_youden['specificity']:.4f}")
print(f"    FN: {m_youden['fn']}, FP: {m_youden['fp']}")

# Find threshold for >= 95% sensitivity
sens_mask = tpr >= 0.95
if sens_mask.any():
    sens95_idx = np.where(sens_mask)[0][-1]
    sens95_t = roc_thresholds[sens95_idx]
    print(f"\n  High-sensitivity threshold ({sens95_t:.4f}, targets >=95% sensitivity):")
    m_sens = compute_metrics(all_labels, all_probs, threshold=sens95_t)
    print(f"    Sensitivity: {m_sens['sensitivity']:.4f}, Specificity: {m_sens['specificity']:.4f}")
    print(f"    FN: {m_sens['fn']}, FP: {m_sens['fp']}")

## 7. Confusion Matrices at Different Operating Points

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

plot_confusion_matrix(all_labels, all_probs, threshold=0.5,
                      title="Default (t=0.5)", ax=axes[0])
plot_confusion_matrix(all_labels, all_probs, threshold=youden_t,
                      title=f"Youden Optimal (t={youden_t:.3f})", ax=axes[1])
if sens_mask.any():
    plot_confusion_matrix(all_labels, all_probs, threshold=sens95_t,
                          title=f"High Sensitivity (t={sens95_t:.3f})", ax=axes[2])

plt.tight_layout()
plt.savefig("../results/confusion_operating_points.png", bbox_inches="tight")
plt.show()

## Summary

This error analysis reveals several important patterns about how the model behaves
in practice. The confidence distribution shows whether errors are borderline or
high-confidence mistakes. The Grad-CAM comparison between correct and misclassified
samples helps identify whether the model relies on clinically valid features or
spurious correlations. The threshold analysis provides concrete guidance for choosing
the right operating point depending on whether the priority is minimizing missed
pneumonia cases (low threshold) or reducing false alarms (high threshold).

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()